# Baselines

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### Carregamento e preparação dos dados

In [9]:
df_train = pd.read_csv("../../dados/treino.csv")
df_val = pd.read_csv("../../dados/validacao.csv")

df_train["date"] = pd.to_datetime(df_train["date"])
df_val["date"] = pd.to_datetime(df_val["date"])

h = 28 
m = 7  
series_names = ["store_total", "FOODS", "HOBBIES"]

In [10]:
previsoes_list = []
val_dates = df_val["date"].values

for col in series_names:
    y_train = df_train[col].values
    T = len(y_train)
    y_last = y_train[-1]
    y_first = y_train[0]
    
    # 1. Média (histórica total)
    yhat_media = np.full(h, np.mean(y_train))
    
    # 2. Naive (último valor)
    yhat_naive = np.full(h, y_last)
    
    # 3. Naive Sazonal (período m=7)
    yhat_sazonal = np.tile(y_train[-m:], h // m)
    
    # 4. Drift (extrapolação linear)
    C = (y_last - y_first) / (T - 1)
    yhat_drift = y_last + np.arange(1, h + 1) * C
    
    modelos = {
        "media": yhat_media,
        "naive": yhat_naive,
        "naive_sazonal": yhat_sazonal,
        "drift": yhat_drift
    }
    
    # Formato longo exigido: date, series, modelo, yhat
    for nome_modelo, yhat in modelos.items():
        df_temp = pd.DataFrame({
            "date": val_dates,
            "series": col,
            "modelo": nome_modelo,
            "yhat": yhat
        })
        previsoes_list.append(df_temp)

df_previsoes_baselines = pd.concat(previsoes_list, ignore_index=True)

In [11]:
metricas_list = []

for col in series_names:
    y_train = df_train[col].values
    y_val = df_val[col].values
    
    # Escala in-sample do MASE: erro médio absoluto do naive sazonal no treino
    escala_mase = np.mean(np.abs(y_train[m:] - y_train[:-m]))
    
    df_prev_serie = df_previsoes_baselines[df_previsoes_baselines["series"] == col]
    
    for modelo in ["media", "naive", "naive_sazonal", "drift"]:
        yhat = df_prev_serie[df_prev_serie["modelo"] == modelo]["yhat"].values
        
        # Erro de previsão
        erro = y_val - yhat
        
        # MAE: Média dos erros absolutos
        mae = np.mean(np.abs(erro))
        
        # RMSE: Raiz quadrada da média dos erros ao quadrado
        rmse = np.sqrt(np.mean(erro**2))
        
        # MASE: MAE da validação / Escala in-sample
        mase = mae / escala_mase
        
        metricas_list.append({
            "series": col,
            "modelo": modelo,
            "mae": mae,
            "rmse": rmse,
            "mase": mase
        })

df_metricas_baselines = pd.DataFrame(metricas_list)
display(df_metricas_baselines)

,series,modelo,mae,rmse,mase
0,store_total,media,813.885203,1109.594703,1.870929
1,store_total,naive,862.750000,966.632280,1.983258
2,store_total,naive_sazonal,459.178571,655.467091,1.055543
3,store_total,drift,863.278662,966.756301,1.984473
4,FOODS,media,526.928571,671.830058,1.540331
5,FOODS,naive,664.642857,722.859451,1.942901
6,FOODS,naive_sazonal,343.857143,420.116140,1.005172
7,FOODS,drift,664.965461,723.256672,1.943844
8,HOBBIES,media,110.994790,141.305168,1.215619
9,HOBBIES,naive,169.035714,204.727606,1.851285
